# Appendix C3. 20-day Rolling Volatility as a Stress Proxy: Sensitivity of LSTM Results to Random Initialization

In [ ]:
# --- ENV FIRST ---
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
# --- IMPORTS ---
import random
import numpy as np
import tensorflow as tf
# --- TF CONFIG ---
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.set_visible_devices([], 'GPU')
# --- RESET ---
tf.keras.backend.clear_session()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import recall_score, f1_score, roc_auc_score, confusion_matrix, accuracy_score, roc_curve
import itertools
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
try:
  import keras_tuner as kt
except:
  !pip install keras-tuner
  import keras_tuner as kt
from google.colab import files
from google.colab import drive
import pandas as pd
import io
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.7 MB/s eta 0:00:00


In [ ]:
def import_data(file_path):
  try:
    drive.mount('/content/drive', force_remount=True)
    # Check if file exists
    if os.path.exists(file_path):
      df = pd.read_parquet(file_path)
      print(f"Loaded dataframe from Drive ({file_path})")
    else:
      raise FileNotFoundError(f"File not found at {file_path}")

  except Exception as e:
    print(f"Drive not available or file missing: {e}")
    print("Please upload dataframe manually.")
    uploaded = files.upload()

    # Automatically read the uploaded file
    file_name = list(uploaded.keys())[0]  # pick the first uploaded file
    try:
      df = pd.read_parquet(io.BytesIO(uploaded[file_name]))
    except:
      print("Wrong file extension. Parquet file required.")
    print(f"Loaded {file_name} from manual upload.")
    return df

In [ ]:
def train_classifier(model, X_train, y_train, X_test, y_test, X_val=False, y_val=False, threshold=0.5, early_stopping=False):
  is_keras = hasattr(model, "fit") and hasattr(model, "predict") and not hasattr(model, "predict_proba")
  if is_keras:
    if early_stopping:
      model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, class_weight=class_weight, validation_data=(X_val, y_val), shuffle=False, callbacks=[combined_metric, es])
    else:
      model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, class_weight=class_weight, validation_data=(X_val, y_val), shuffle=False)
    y_score = model.predict(X_test).ravel()
    y_pred = (y_score >= threshold).astype(int)
  else:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

  # Output Following Metrics:
  recall = recall_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  roc_auc = roc_auc_score(y_test, y_score)
  cm = confusion_matrix(y_test, y_pred)

  return recall, f1, roc_auc, cm, y_score, y_pred

In [ ]:
file_path = "..."
df_model_2 = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_model_2.parquet to df_model_2.parquet
Loaded df_model_2.parquet from manual upload.


In [ ]:
# align the dataframe at the same starting date of the VSRI
df_model_2 = df_model_2.loc['2017-01-03':].dropna()

In [ ]:
df_model_2.shape

(2384, 2)

In [ ]:
# split data in training and test set and apply purging
split_date = "2024-03-31"
purge = 20
train_2 = df_model_2.loc[:split_date].iloc[:-purge]
test_2  = df_model_2[df_model_2.index > split_date]
X_train_2 = train_2[["vn_volatility"]]
y_train_2 = train_2["systemic_stress_event"]
X_test_2  = test_2[["vn_volatility"]]
y_test_2  = test_2["systemic_stress_event"]

In [ ]:
timesteps = 60
split_idx = pd.Timestamp("2023-03-01")
X_train_lstm_bench_2 = X_train_2[X_train_2.index < split_idx].iloc[:-purge]
y_train_lstm_bench_2 = y_train_2[y_train_2.index < split_idx].iloc[:-purge]
X_val_lstm_bench_2 = X_train_2.loc[split_idx:]
y_val_lstm_bench_2 = y_train_2.loc[split_idx:]

X_train_values_bench_2 = X_train_lstm_bench_2.values
y_train_values_bench_2 = y_train_lstm_bench_2.values
X_test_values_bench_2 = X_test_2.values
y_test_values_bench_2 = y_test_2.values
X_val_values_bench_2 = X_val_lstm_bench_2.values
y_val_values_bench_2 = y_val_lstm_bench_2.values
X_train_seq_bench_2 = []
y_train_seq_bench_2 = []
X_test_seq_bench_2 = []
y_test_seq_bench_2 = []
X_val_seq_bench_2 = []
y_val_seq_bench_2 = []
for i in range(timesteps, len(X_train_values_bench_2)):
  X_train_seq_bench_2.append(X_train_values_bench_2[i - timesteps:i])
  y_train_seq_bench_2.append(y_train_values_bench_2[i])
for i in range(timesteps, len(X_test_values_bench_2)):
  X_test_seq_bench_2.append(X_test_values_bench_2[i - timesteps:i])
  y_test_seq_bench_2.append(y_test_values_bench_2[i])
for i in range(timesteps, len(X_val_values_bench_2)):
  X_val_seq_bench_2.append(X_val_values_bench_2[i - timesteps:i])
  y_val_seq_bench_2.append(y_val_values_bench_2[i])

X_train_seq_bench_2 = np.array(X_train_seq_bench_2)
y_train_seq_bench_2 = np.array(y_train_seq_bench_2)
X_test_seq_bench_2 = np.array(X_test_seq_bench_2)
y_test_seq_bench_2 = np.array(y_test_seq_bench_2)
X_val_seq_bench_2 = np.array(X_val_seq_bench_2)
y_val_seq_bench_2 = np.array(y_val_seq_bench_2)

In [ ]:
print(X_train_seq_bench_2.shape,
      y_train_seq_bench_2.shape,
      X_val_seq_bench_2.shape,
      y_val_seq_bench_2.shape,
      X_test_seq_bench_2.shape,
      y_test_seq_bench_2.shape)

(1522, 60, 1) (1522,) (202, 60, 1) (202,) (440, 60, 1) (440,)


In [ ]:
class CombinedMetric(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    logs = logs or {}
    recall = logs.get("val_recall", 0)
    pr_auc = logs.get("val_pr_auc", 0)
    logs["val_combined"] = 0.3 * recall + 0.7 * pr_auc

In [ ]:
classes = np.unique(y_train_seq_bench_2)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_seq_bench_2)
class_weight = dict(zip(classes, weights))
epochs=200
batch_size=32

In [ ]:
seeds = [38, 60, 96, 146, 162]
robustness_results_bench_2 = []
input_shape = X_train_seq_bench_2.shape[1:]

In [ ]:
for seed in seeds:
  print(f"\nRunning seed {seed}")
  tf.keras.backend.clear_session()
  random.seed(seed)
  np.random.seed(seed)
  tf.keras.utils.set_random_seed(seed)

  # LSTM MODEL
  lstm_model_bench_2 = keras.Sequential()
  lstm_model_bench_2.add(layers.LSTM(128, return_sequences=True, input_shape=input_shape))
  lstm_model_bench_2.add(layers.Dropout(0.5))
  lstm_model_bench_2.add(layers.LSTM(128, return_sequences=True))
  lstm_model_bench_2.add(layers.LSTM(16, return_sequences=True))
  lstm_model_bench_2.add(layers.LSTM(8, return_sequences=True))
  lstm_model_bench_2.add(layers.Dropout(0.3))
  lstm_model_bench_2.add(layers.LSTM(8, return_sequences=False))
  lstm_model_bench_2.add(layers.Dropout(0.3))
  # Dense layers
  lstm_model_bench_2.add(layers.Dense(16, activation="relu"))
  lstm_model_bench_2.add(layers.Dense(1, activation="sigmoid"))
  # Compile
  lstm_model_bench_2.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                             loss="binary_crossentropy",
                             metrics=[keras.metrics.AUC(name="pr_auc", curve="PR"),
                                      keras.metrics.AUC(name="roc_auc", curve="ROC"),
                                      tf.keras.metrics.Recall(name="recall"),
                                      tf.keras.metrics.Precision(name="precision")])

  combined_metric = CombinedMetric()
  es = tf.keras.callbacks.EarlyStopping(monitor="val_combined",
                                        mode="max",
                                        patience=50,
                                        restore_best_weights=True,
                                        verbose=1)

  # TRAIN
  lstm_results_bench_2 = train_classifier(lstm_model_bench_2, X_train_seq_bench_2, y_train_seq_bench_2, X_test_seq_bench_2, y_test_seq_bench_2, X_val=X_val_seq_bench_2, y_val=y_val_seq_bench_2, early_stopping=True)
  lstm_recall_bench_2, lstm_f1_bench_2, lstm_roc_auc_bench_2, lstm_cm_bench_2, lstm_probs_bench_2, lstm_preds_bench_2 = lstm_results_bench_2
  # Store results
  robustness_results_bench_2.append({"seed": seed,
                                     "recall": lstm_recall_bench_2,
                                     "f1": lstm_f1_bench_2,
                                     "roc_auc": lstm_roc_auc_bench_2,
                                     "tn": lstm_cm_bench_2[0,0],
                                     "fp": lstm_cm_bench_2[0,1],
                                     "fn": lstm_cm_bench_2[1,0],
                                     "tp": lstm_cm_bench_2[1,1]})


Running seed 38


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 33s 223ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0675 - recall: 0.5354 - roc_auc: 0.5000 - val_loss: 0.6940 - val_pr_auc: 0.0495 - val_precision: 0.0495 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3347
Epoch 2/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 11s 221ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0762 - recall: 0.8110 - roc_auc: 0.5000 - val_loss: 0.6937 - val_pr_auc: 0.0495 - val_precision: 0.0495 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3347
Epoch 3/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 202ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0816 - recall: 0.8819 - roc_auc: 0.5000 - val_loss: 0.6935 - val_pr_auc: 0.0495 - val_precision: 0.0495 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3347
Epoch 4/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 195ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0703 - recall: 0.6929 - roc_auc: 0.5000 - val_loss: 0.6933 - val_pr_auc: 0.0495 - val_p

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 20s 227ms/step - loss: 0.6935 - pr_auc: 0.0725 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.4409 - val_loss: 0.6882 - val_pr_auc: 0.0495 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0347
Epoch 2/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 185ms/step - loss: 0.6935 - pr_auc: 0.0749 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.4600 - val_loss: 0.6887 - val_pr_auc: 0.0552 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5547 - val_combined: 0.0387
Epoch 3/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 190ms/step - loss: 0.6933 - pr_auc: 0.0757 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.4610 - val_loss: 0.6888 - val_pr_auc: 0.0495 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0347
Epoch 4/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 203ms/step - loss: 0.6933 - pr_auc: 0.0744 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc:

...

Epoch 1/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 20s 229ms/step - loss: 0.6936 - pr_auc: 0.0725 - precision: 0.0270 - recall: 0.0079 - roc_auc: 0.4431 - val_loss: 0.6887 - val_pr_auc: 0.0645 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.6224 - val_combined: 0.0452
Epoch 2/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 185ms/step - loss: 0.6933 - pr_auc: 0.0778 - precision: 0.0769 - recall: 0.0236 - roc_auc: 0.4736 - val_loss: 0.6886 - val_pr_auc: 0.0323 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.2943 - val_combined: 0.0226
Epoch 3/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 193ms/step - loss: 0.6933 - pr_auc: 0.0799 - precision: 0.0851 - recall: 0.0315 - roc_auc: 0.4840 - val_loss: 0.6887 - val_pr_auc: 0.0704 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.6562 - val_combined: 0.0493
Epoch 4/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 203ms/step - loss: 0.6931 - pr_auc: 0.0858 - precision: 0.0727 - recall: 0.0315 - roc_auc: 0.5094 - val_loss: 0.6879 - val

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 19s 215ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.5000 - val_loss: 0.6917 - val_pr_auc: 0.0495 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0347
Epoch 2/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 189ms/step - loss: 0.6932 - pr_auc: 0.0834 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.5000 - val_loss: 0.6916 - val_pr_auc: 0.0495 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0347
Epoch 3/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 203ms/step - loss: 0.6932 - pr_auc: 0.0834 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.5000 - val_loss: 0.6918 - val_pr_auc: 0.0495 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5000 - val_combined: 0.0347
Epoch 4/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 202ms/step - loss: 0.6932 - pr_auc: 0.0834 - precision: 0.1111 - recall: 0.0236 - roc_auc: 0.5000

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 20s 230ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0569 - recall: 0.2756 - roc_auc: 0.5000 - val_loss: 0.6933 - val_pr_auc: 0.0495 - val_precision: 0.0495 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3347
Epoch 2/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 187ms/step - loss: 0.6932 - pr_auc: 0.0834 - precision: 0.0660 - recall: 0.3937 - roc_auc: 0.5000 - val_loss: 0.6933 - val_pr_auc: 0.0495 - val_precision: 0.0495 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3347
Epoch 3/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 191ms/step - loss: 0.6932 - pr_auc: 0.0834 - precision: 0.0809 - recall: 0.5827 - roc_auc: 0.5000 - val_loss: 0.6937 - val_pr_auc: 0.0495 - val_precision: 0.0495 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3347
Epoch 4/200
48/48 ━━━━━━━━━━━━━━━━━━━━ 10s 204ms/step - loss: 0.6933 - pr_auc: 0.0834 - precision: 0.0758 - recall: 0.5591 - roc_auc: 0.5000 - val_loss: 0.6928 - val_pr_auc: 0.0495 - val_pr

In [ ]:
df_robustness_bench_2 = pd.DataFrame(robustness_results_bench_2)
df_robustness_bench_2 = pd.concat([df_robustness_bench_2,pd.DataFrame({'seed': 23,
                                                                       'recall': 1,
                                                                       'f1': 0.11,
                                                                       'roc_auc': 0.69,
                                                                       'tn': 0,
                                                                       'fp': 415,
                                                                       'fn': 0,
                                                                       'tp': 25},
                                                                      index=[0])], ignore_index=True) #add results from baseline model
df_robustness_bench_2

,seed,recall,f1,roc_auc,tn,fp,fn,tp
0,38,1.0,0.107527,0.276145,0,415,0,25
1,60,0.0,0.000000,0.725494,415,0,25,0
2,96,0.0,0.000000,0.703373,415,0,25,0
3,146,0.0,0.000000,0.272964,415,0,25,0
4,162,1.0,0.107527,0.297060,0,415,0,25
5,23,1.0,0.110000,0.690000,0,415,0,25


```python
df_robustness_bench_2.to_csv("df_robustness_bench_2.csv", index=True)
files.download("df_robustness_bench_2.csv")
df_robustness_bench_2.to_parquet("df_robustness_bench_2.parquet", index=True)
files.download("df_robustness_bench_2.parquet")
```

In [ ]:
#df_robustness_bench_2 = import_data(file_path)

In [ ]:
df_robustness_bench_2 = df_robustness_bench_2.set_index('seed')[['recall',	'f1',	'roc_auc']].sort_index()
df_robustness_bench_2 = df_robustness_bench_2.rename(columns={'recall':'Recall',	'f1':'F1-Score',	'roc_auc':'ROC-AUC'})

In [ ]:
df_robustness_bench_2.round(2)

,Recall,F1-Score,ROC-AUC
seed,,,
23,1.0,0.11,0.69
38,1.0,0.11,0.28
60,0.0,0.00,0.73
96,0.0,0.00,0.70
146,0.0,0.00,0.27
162,1.0,0.11,0.30


In [ ]:
df_robustness_bench_2.mean().round(2).rename('mean')

,mean
Recall,0.50
F1-Score,0.05
ROC-AUC,0.49


In [ ]:
df_robustness_bench_2.std().round(2).rename('std')

,std
Recall,0.55
F1-Score,0.06
ROC-AUC,0.23
